# 4. Query OpenStreetMap with OSMnx — and save the result

OSMnx provides a Python interface for downloading OpenStreetMap features.

Official OSMnx documentation describes `features_from_place()` as returning OpenStreetMap features within the boundary of a named place as a GeoDataFrame.

This notebook performs the live query and then **saves the raw response to disk** so later notebooks can reopen it.

In [ ]:
from pathlib import Path

import geopandas as gpd
import osmnx as ox

## Create the folders we need

`Path.mkdir()` creates a directory from Python.

`parents=True` creates any missing parent folders.

`exist_ok=True` means Python will not raise an error if the folder already exists.

In [ ]:
raw_dir = Path("../data/raw")
processed_dir = Path("../data/processed")
output_dir = Path("../output")

raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

## Define the query

Keeping important settings in variables makes the workflow easy to adapt.

In [ ]:
place = "Toronto, Ontario, Canada"

tags = {
    "amenity": "library"
}

## Download the features

This cell requires an internet connection and a functioning OpenStreetMap/Overpass service.

Do not repeatedly rerun a successful request. Once it succeeds, save the response and use the saved file.

### Safe classroom fallback

A successful live OSM download is already included at:

```text
../data/backup/toronto_osm_libraries_live_backup.gpkg
```

The code below uses a simple `USE_LIVE_QUERY` switch. Set it to `True` for the live query or `False` to use the protected classroom backup.

The backup is kept in a separate directory so it is not overwritten by a new live request.


In [ ]:
USE_LIVE_QUERY = True

backup_osm_file = Path(
    "../data/backup/toronto_osm_libraries_live_backup.gpkg"
)

if USE_LIVE_QUERY:
    libraries = ox.features_from_place(
        place,
        tags=tags
    )
    print("Using live OSM data.")
else:
    libraries = gpd.read_file(
        backup_osm_file
    )
    print("Using the saved classroom backup.")

print("Rows available:", len(libraries))

## Inspect before saving

OSMnx commonly returns a GeoDataFrame whose index contains the OSM element type and OSM ID.

In [ ]:
display(libraries.head())

print("CRS:", libraries.crs)
print("\nGeometry types:")
print(libraries.geometry.geom_type.value_counts())

## Save the raw query result

GeoPackage is useful here because it preserves geometry and attributes in one file.

Later notebooks will reopen this file rather than relying on this notebook's Python variables.

In [ ]:
raw_osm_file = raw_dir / "toronto_osm_libraries.gpkg"

# Only overwrite the normal raw-data file when the live query
# actually succeeded. The protected backup is never overwritten.
if used_live_query:
    libraries.to_file(
        raw_osm_file,
        driver="GPKG"
    )
    print("Saved new live result:", raw_osm_file)
else:
    print(
        "Using the protected backup; "
        "the existing raw file was left unchanged."
    )

## If the live query fails

Use the practice file:

```text
../data/practice/toronto_library_points_practice.geojson
```

That file comes from the supplied Toronto Public Library course data and is **not** represented as an OpenStreetMap extract. It exists only so downstream spatial-analysis practice can continue.